# GTFS Scheduled Trips and Stops Analysis

This script processes GTFS data to analyze transit service for a selection of agencies across different days of the week. The main steps are:
1. Filter trips by agency and date: Selects scheduled trips for the specified agencies on a Sunday, Saturday, and a weekday.
2. Extract feed keys and retrieve stops: For each day, unique feed keys from the trips are used to query the corresponding daily scheduled stops, including stop identifiers, names, and associated route information.
3. Expand route-stop relationships: Many stops are served by multiple routes. The script transforms the stops data so that each row represents a unique stop × route combination.
4. Merge with trip counts: Trip-level data is aggregated by feed and route to calculate the number of trips per route. This information is merged with the expanded stops data, providing trip counts for each stop × route combination.
5. Aggregate stop-level statistics: Stops are summarized by feed, stop ID, stop name, and route type to calculate:
- Total number of trips serving each stop.
- Number of distinct routes serving each stop.

The final dataset provides a detailed overview of transit service at the stop level, including how many trips and routes serve each stop for the selected agencies and dates.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
import datetime as dt
from calitp_data_analysis.sql import get_engine
from shared_utils import gtfs_utils_v2
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
# GCS FILE PATH
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
agencies_to_find = [
    "Bay Area 511 BART Schedule",
    "Big Blue Bus Schedule",
    "Caltrain Schedule",
    "Culver City Schedule",
    "Foothill Schedule",
    "Fresno Schedule",
    "Gold Coast Schedule",
    "Golden Gate Park Shuttle Schedule",
    "Bay Area 511 Golden Gate Transit Schedule",
    "Long Beach Schedule",
    "OCTA Schedule",
    "OmniTrans Schedule",
    "Riverside Schedule",
    "Sacramento Schedule",
    "Santa Cruz Schedule",
    "SBMTD Schedule",
    "SamTrans Schedule",
    "San Diego Schedule"
]

agency_sql = ', '.join(f"'{agency}'" for agency in agencies_to_find)

In [5]:
trip_columns = [
    "feed_key", "trip_instance_key", "gtfs_dataset_key", "trip_id", "route_id", 
    "route_type", "name", "route_short_name", "route_long_name", "service_date", 
    "trip_first_departure_datetime_local_tz", "trip_last_arrival_datetime_local_tz",
    "time_of_day", "service_hours", "num_stop_times", "num_distinct_stops_served",
    "frequencies_defined_trip", "is_gtfs_flex_trip", "is_entirely_demand_responsive_trip"
]

In [6]:
with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-18')
          AND name IN ({agency_sql})
    """
    trips_sunday_data = pd.read_sql(query, connection)



with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-21')
          AND name IN ({agency_sql})
    """
    trips_weekday_data = pd.read_sql(query, connection)


with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-17')
          AND name IN ({agency_sql})
    """
    trips_saturday_data = pd.read_sql(query, connection)


In [7]:
# Get unique feed_keys from your trips_weekday_data
weekday_feed_key = trips_weekday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_weekday = ",".join(f"'{fk}'" for fk in weekday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-21')
          AND feed_key IN ({feed_keys_str_weekday})
    """
    stops_unique_weekday = pd.read_sql(query, connection)

In [8]:
# Get unique feed_keys from your trips_weekday_data
saturday_feed_key = trips_saturday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_saturday = ",".join(f"'{fk}'" for fk in saturday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str_saturday})
    """
    stops_unique_saturday = pd.read_sql(query, connection)

In [9]:
# Get unique feed_keys from your trips_weekday_data
sunday_feed_key = trips_sunday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_sunday = ",".join(f"'{fk}'" for fk in sunday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str_sunday})
    """
    stops_unique_sunday = pd.read_sql(query, connection)

In [10]:
stops_unique_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27362 entries, 0 to 27361
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   feed_key            27362 non-null  object             
 1   stop_id             27362 non-null  object             
 2   _feed_valid_from    27362 non-null  datetime64[ns, UTC]
 3   n_hours_in_service  27362 non-null  int64              
 4   arrivals_early_am   27362 non-null  int64              
 5   arrivals_am_peak    27362 non-null  int64              
 6   arrivals_midday     27362 non-null  int64              
 7   arrivals_pm_peak    27362 non-null  int64              
 8   arrivals_evening    27362 non-null  int64              
 9   route_id_array      27362 non-null  object             
 10  route_type_array    27362 non-null  object             
 11  stop_key            27362 non-null  object             
 12  tts_stop_name       0 non-null  

In [11]:
trips_weekday_data.head(5)

,feed_key,trip_instance_key,gtfs_dataset_key,trip_id,route_id,route_type,name,route_short_name,route_long_name,service_date,trip_first_departure_datetime_local_tz,trip_last_arrival_datetime_local_tz,time_of_day,service_hours,num_stop_times,num_distinct_stops_served,frequencies_defined_trip,is_gtfs_flex_trip,is_entirely_demand_responsive_trip
0,f6774d861953d4f4cdcffec95e2652c7,e042cf31b79f52e465573830a8d27a86,4b317fc27dde351e12253d46cedd8df0,1044090,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-21,2025-05-21 17:27:00,2025-05-21 18:20:00,pm_peak,0.883333,42,42,False,False,False
1,f6774d861953d4f4cdcffec95e2652c7,14a723ef3792c42be486d5bb4377e929,4b317fc27dde351e12253d46cedd8df0,1616090,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-21,2025-05-21 12:12:00,2025-05-21 12:55:00,midday,0.716667,42,42,False,False,False
2,f6774d861953d4f4cdcffec95e2652c7,55ee745c3441ec52ea8a6c6a1f340292,4b317fc27dde351e12253d46cedd8df0,1528090,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-21,2025-05-21 19:05:00,2025-05-21 19:49:00,pm_peak,0.733333,42,42,False,False,False
3,f6774d861953d4f4cdcffec95e2652c7,572fb7554bab4cebc9e4541eaffac2c4,4b317fc27dde351e12253d46cedd8df0,68090,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-21,2025-05-21 17:42:00,2025-05-21 18:33:00,pm_peak,0.850000,42,42,False,False,False
4,f6774d861953d4f4cdcffec95e2652c7,37662aafeda8993d1b472a675ef21e5b,4b317fc27dde351e12253d46cedd8df0,285090,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-21,2025-05-21 13:57:00,2025-05-21 14:43:00,midday,0.766667,42,42,False,False,False


In [12]:
def trips_per_route(trips_df):
    """
    Aggregate trips per feed and route.

    Parameters:
    - trips_df: pd.DataFrame with trip-level data

    Returns:
    - pd.DataFrame with n_trips per feed × route, plus first found metadata
    """
    return trips_df.groupby(
        ['feed_key', 'route_id', 'gtfs_dataset_key', 'name', 'route_type'],
        as_index=False
    ).agg(
        n_trips=('trip_id', 'nunique'),
        route_long_name= ('route_long_name', 'first')
    )

In [13]:
# trips per feed_key × route_id
trips_per_route_weekday = trips_per_route(trips_weekday_data)
trips_per_route_saturday = trips_per_route(trips_saturday_data)
trips_per_route_sunday = trips_per_route(trips_sunday_data)

In [14]:
trips_per_route_weekday.head(5)

,feed_key,route_id,gtfs_dataset_key,name,route_type,n_trips,route_long_name
0,1eb334a547f5c2cdbdce3f7cc7b03e2b,001,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,3,124,GREENBACK
1,1eb334a547f5c2cdbdce3f7cc7b03e2b,010,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,3,28,FSL Route 10
2,1eb334a547f5c2cdbdce3f7cc7b03e2b,011,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,3,48,NATOMAS/LAND PARK
3,1eb334a547f5c2cdbdce3f7cc7b03e2b,013,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,3,40,NATOMAS/ARDEN
4,1eb334a547f5c2cdbdce3f7cc7b03e2b,015,98588c1726b50acbef430783ab0e734e,Sacramento Schedule,3,56,DEL PASO HEIGHTS


In [15]:
# Found that stop_id in BART is different for each platform so combining them before. 

# List of dataframes to process
stops_dfs = [stops_unique_weekday, stops_unique_saturday, stops_unique_sunday]

# Target feed_key to collapse platform stop_ids
target_feed_key = '4e549244dd7ce383a3f4337f00134b27'

# Apply in-place transformation to all dataframes
for df in stops_dfs:
    mask = df['feed_key'] == target_feed_key
    df.loc[mask, 'stop_id'] = df.loc[mask, 'stop_id'].astype(str).str[:-1]

In [27]:
def expand_routes_fixed(row):
    route_ids = row['route_id_array'] if isinstance(row['route_id_array'], list) else []
    route_types = row['route_type_array'] if isinstance(row['route_type_array'], list) else []
    
    if len(route_types) == 1 and len(route_ids) > 1:
        route_types = route_types * len(route_ids)
    
    pairs = list(zip(route_ids, route_types))
    stop_code_value = row.get('stop_code', pd.NA)
    
    # Create geometry point if lat/lon available
    if 'stop_lat' in row and 'stop_lon' in row and pd.notna(row['stop_lat']) and pd.notna(row['stop_lon']):
        pt_geom = Point(row['stop_lon'], row['stop_lat'])
    else:
        pt_geom = pd.NA

    # # Calculate arrivals_all_day
    # arrival_cols = ['arrivals_early_am', 'arrivals_am_peak', 'arrivals_midday', 'arrivals_pm_peak', 'arrivals_evening']
    # arrivals_all_day = sum(row.get(col, 0) or 0 for col in arrival_cols)
    
    if not pairs:
        return pd.DataFrame({
            'feed_key': [row['feed_key']],
            'stop_id': [row['stop_id']],
            'stop_code': [stop_code_value],
            'stop_name': [row['stop_name']],
            'route_id': [pd.NA],
            'route_type': [pd.NA],
            'pt_geom': [pt_geom],
            # 'arrivals_all_day': [arrivals_all_day]
        })
    
    return pd.DataFrame({
        'feed_key': [row['feed_key']] * len(pairs),
        'stop_id': [row['stop_id']] * len(pairs),
        'stop_code': [stop_code_value] * len(pairs),
        'stop_name': [row['stop_name']] * len(pairs),
        'route_id': [r[0] for r in pairs],
        'route_type': [r[1] for r in pairs],
        'pt_geom': [pt_geom_value for _ in pairs],
        # 'arrivals_all_day': [arrivals_all_day] * len(pairs)
    })

In [29]:
def expand_routes_fixed(row):
    """
    Expands a stop row with multiple route_ids and route_types into separate rows
    for each route, keeping existing pt_geom and stop_code.

    Parameters
    ----------
    row : pd.Series
        A row containing:
        - route_id_array: list of route IDs
        - route_type_array: list of route types
        - stop_code
        - pt_geom
        - feed_key, stop_id, stop_name

    Returns
    -------
    pd.DataFrame
        Expanded rows for each route_id × route_type pair.
    """
    # Ensure arrays exist
    route_ids = row['route_id_array'] if isinstance(row['route_id_array'], list) else []
    route_types = row['route_type_array'] if isinstance(row['route_type_array'], list) else []

    # If only one route_type but multiple route_ids, replicate route_type
    if len(route_types) == 1 and len(route_ids) > 1:
        route_types = route_types * len(route_ids)

    # Pair routes with types
    pairs = list(zip(route_ids, route_types))
    stop_code_value = row.get('stop_code', pd.NA)
    pt_geom_value = row.get('pt_geom', pd.NA)

    # # Calculate arrivals_all_day
    # arrival_cols = ['arrivals_early_am', 'arrivals_am_peak', 'arrivals_midday', 'arrivals_pm_peak', 'arrivals_evening']
    # arrivals_all_day = sum(row.get(col, 0) or 0 for col in arrival_cols)

    # Handle case with no routes
    if not pairs:
        return pd.DataFrame({
            'feed_key': [row['feed_key']],
            'stop_id': [row['stop_id']],
            'stop_code': [stop_code_value],
            'stop_name': [row['stop_name']],
            'route_id': [pd.NA],
            'route_type': [pd.NA],
            'pt_geom': [pt_geom_value],
        })

    # Expand into multiple rows
    return pd.DataFrame({
        'feed_key': [row['feed_key']] * len(pairs),
        'stop_id': [row['stop_id']] * len(pairs),
        'stop_code': [stop_code_value] * len(pairs),
        'stop_name': [row['stop_name']] * len(pairs),
        'route_id': [r[0] for r in pairs],
        'route_type': [r[1] for r in pairs],
        'pt_geom': [pt_geom_value for _ in pairs],
        # 'arrivals_all_day': [arrivals_all_day] * len(pairs)
    })

In [30]:
stops_expanded_weekday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_weekday.iterrows()], ignore_index=True)
stops_expanded_saturday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_saturday.iterrows()], ignore_index=True)
stops_expanded_sunday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_sunday.iterrows()], ignore_index=True)

In [32]:
def merge_and_aggregate_stops_listcodes_with_nroutes(stops_expanded_df, trips_per_route_df):
    """
    Merge expanded stops with trip counts and aggregate stop × route-level statistics,
    combining multiple stop_codes into a list if they exist.
    Adds n_routes per stop × route group.

    Parameters:
    - stops_expanded_df: pd.DataFrame with stop × route rows (expanded)
    - trips_per_route_df: pd.DataFrame with number of trips per feed_key × route_id × route_type

    Returns:
    - pd.DataFrame aggregated by feed_key, stop_id, stop_name, route_id, route_type
      with total trips per stop × route, stop_codes as lists if multiple exist,
      and n_routes indicating number of unique route_ids per group
    """
    # Ensure route_type and route_id columns have the same type
    stops_expanded_df['route_type'] = stops_expanded_df['route_type'].astype(str)
    trips_per_route_df['route_type'] = trips_per_route_df['route_type'].astype(str)
    stops_expanded_df['route_id'] = stops_expanded_df['route_id'].astype(str)
    trips_per_route_df['route_id'] = trips_per_route_df['route_id'].astype(str)

    # Merge stops with trips
    merged = pd.merge(
        stops_expanded_df,
        trips_per_route_df,
        on=['feed_key', 'route_id', 'route_type'],
        how='left'
    )

    # Fill NaN for missing trips
    merged['n_trips'] = merged['n_trips'].fillna(0)

    # Aggregate per stop × route
    stops_aggregated = merged.groupby(
        ['feed_key', 'stop_id', 'stop_name', 'route_id', 'route_type'],
        dropna=False
    ).agg(
        n_trips=('n_trips', 'sum'),
        # arrivals_all_day = ('arrivals_all_day', 'sum'),
        n_routes=('route_id', 'nunique'),  # number of unique route_ids per group
        gtfs_dataset_key=('gtfs_dataset_key', 'first'),
        name=('name', 'first'),
        route_long_name=('route_long_name', 'first'),
        stop_code=('stop_code', 'first'),
        pt_geom=('pt_geom', lambda x: next((v for v in x if pd.notna(v)), pd.NA))).reset_index()

    return stops_aggregated

In [33]:
def merge_and_aggregate_without_routes(stops_expanded_df, trips_per_route_df):
    # Type alignment
    stops_expanded_df['route_type'] = stops_expanded_df['route_type'].astype(str)
    trips_per_route_df['route_type'] = trips_per_route_df['route_type'].astype(str)
    stops_expanded_df['route_id'] = stops_expanded_df['route_id'].astype(str)
    trips_per_route_df['route_id'] = trips_per_route_df['route_id'].astype(str)

    # Merge
    merged = pd.merge(
        stops_expanded_df,
        trips_per_route_df,
        on=['feed_key', 'route_id', 'route_type'],
        how='left'
    )

    merged['n_trips'] = merged['n_trips'].fillna(0)

    # Aggregate (WITHOUT route)
    agg = merged.groupby(
        ['feed_key', 'stop_id', 'stop_name'],
        dropna=False
    ).agg(
        n_trips=('n_trips', 'sum'),
        # arrivals_all_day=('arrivals_all_day', 'sum'),
        n_routes=('route_id', 'nunique'),
        gtfs_dataset_key=('gtfs_dataset_key', 'first'),
        name=('name', 'first'),
        stop_code=('stop_code', 'first'),
        pt_geom=('pt_geom', lambda x: next((v for v in x if pd.notna(v)), pd.NA))).reset_index()


    # Add route columns for schema consistency
    agg['route_id'] = None
    agg['route_type'] = None
    agg['route_long_name'] = None

    return agg

In [34]:
feeds_with_route = [
    '7a3f513c343b16a30c135ed7d332b6d6',  # Big Blue Bus Schedule
    'f6774d861953d4f4cdcffec95e2652c7',  # Culver City Schedule
    '661ef844bdaa253e8b950740f76061b1',  # Foothill Schedule
    '3cb676436aad669e52042c0e97a9a051',  # Gold Coast Schedule
    'de77cb40e92fb47fa8d16228977cfb86',  # Golden Gate Transit Schedule
    'cddd375786d835389a7beb9632369907',  # Long Beach Schedule
    'cd299184726656597ae2cdb4f4e81e4a',  # OCTA Schedule
    '209e55541803479b23ebb6eea4fae5fa',  # OmniTrans Schedule
    '6eb2b575bee157dace7a2c7155d3cb25',  # Riverside Schedule
    '1eb334a547f5c2cdbdce3f7cc7b03e2b',  # Sacramento Schedule (SacRT)
    'db97cc02836aa5f0cf647d80160b23ec',  # SamTrans Schedule
    '1fff52f9349da228c56eef492df5001b'   # San Diego Schedule (SDMTS)
]


feeds_without_route = [
    '4e549244dd7ce383a3f4337f00134b27',  # BART Schedule
    'f189d5677d4a106b98585f3c5d4fd42c',  # Caltrain Schedule
    '23d1893801eefadf7544a670a3bcd312',  # Fresno Schedule
    'ea33d4691b573336fc9c43c23fa90f65',  # Golden Gate Park Shuttle
    '5a075de618b4d2d2383550863fc8e44e',  # Santa Cruz Schedule
    '6b5c8acdaa4dcb280591578fcbf6c18e'   # SBMTD Schedule
]

In [35]:
# ------------------------
# WEEKDAY
# ------------------------
stops_with_route_weekday = stops_expanded_weekday[
    stops_expanded_weekday['feed_key'].isin(feeds_with_route)
].copy()
stops_without_route_weekday = stops_expanded_weekday[
    stops_expanded_weekday['feed_key'].isin(feeds_without_route)
].copy()

# Aggregate
agg_with_route_weekday = merge_and_aggregate_stops_listcodes_with_nroutes(
    stops_with_route_weekday, trips_per_route_weekday
)
agg_without_route_weekday = merge_and_aggregate_without_routes(
    stops_without_route_weekday, trips_per_route_weekday
)

# Combine results
stops_aggregated_weekday = pd.concat(
    [agg_with_route_weekday, agg_without_route_weekday],
    ignore_index=True
)

In [36]:
stops_aggregated_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36383 entries, 0 to 36382
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   feed_key          36383 non-null  object
 1   stop_id           36383 non-null  object
 2   stop_name         36383 non-null  object
 3   route_id          33395 non-null  object
 4   route_type        33395 non-null  object
 5   n_trips           36383 non-null  int64 
 6   n_routes          36383 non-null  int64 
 7   gtfs_dataset_key  36383 non-null  object
 8   name              36383 non-null  object
 9   route_long_name   32361 non-null  object
 10  stop_code         32888 non-null  object
 11  pt_geom           36383 non-null  object
dtypes: int64(2), object(10)
memory usage: 3.3+ MB


In [37]:
# ------------------------
# SATURDAY
# ------------------------
stops_with_route_saturday = stops_expanded_saturday[
    stops_expanded_saturday['feed_key'].isin(feeds_with_route)
].copy()
stops_without_route_saturday = stops_expanded_saturday[
    stops_expanded_saturday['feed_key'].isin(feeds_without_route)
].copy()

stops_with_route_saturday['route_type'] = stops_with_route_saturday['route_type'].astype(str)
stops_with_route_saturday['route_id'] = stops_with_route_saturday['route_id'].astype(str)
stops_without_route_saturday['route_type'] = stops_without_route_saturday['route_type'].astype(str)
stops_without_route_saturday['route_id'] = stops_without_route_saturday['route_id'].astype(str)

agg_with_route_saturday = merge_and_aggregate_stops_listcodes_with_nroutes(
    stops_with_route_saturday, trips_per_route_saturday
)
agg_without_route_saturday = merge_and_aggregate_without_routes(
    stops_without_route_saturday, trips_per_route_saturday
)

stops_aggregated_saturday = pd.concat(
    [agg_with_route_saturday, agg_without_route_saturday],
    ignore_index=True
)

In [38]:
# ------------------------
# SUNDAY
# ------------------------
stops_with_route_sunday = stops_expanded_sunday[
    stops_expanded_sunday['feed_key'].isin(feeds_with_route)
].copy()
stops_without_route_sunday = stops_expanded_sunday[
    stops_expanded_sunday['feed_key'].isin(feeds_without_route)
].copy()

stops_with_route_sunday['route_type'] = stops_with_route_sunday['route_type'].astype(str)
stops_with_route_sunday['route_id'] = stops_with_route_sunday['route_id'].astype(str)
stops_without_route_sunday['route_type'] = stops_without_route_sunday['route_type'].astype(str)
stops_without_route_sunday['route_id'] = stops_without_route_sunday['route_id'].astype(str)

agg_with_route_sunday = merge_and_aggregate_stops_listcodes_with_nroutes(
    stops_with_route_sunday, trips_per_route_sunday
)
agg_without_route_sunday = merge_and_aggregate_without_routes(
    stops_without_route_sunday, trips_per_route_sunday
)

stops_aggregated_sunday = pd.concat(
    [agg_with_route_sunday, agg_without_route_sunday],
    ignore_index=True
)

In [39]:
# ------------------------
# ALL DAY
# ------------------------

stops_expanded_all_days = pd.concat(
    [stops_expanded_weekday, stops_expanded_saturday, stops_expanded_sunday],
    ignore_index=True
)

stops_with_route_all_day = stops_expanded_all_days[
    stops_expanded_all_days['feed_key'].isin(feeds_with_route)
].copy()

stops_without_route_all_day = stops_expanded_all_days[
    stops_expanded_all_days['feed_key'].isin(feeds_without_route)
].copy()


all_day_with_routes = merge_and_aggregate_stops_listcodes_with_nroutes(
    stops_with_route_all_day, 
    pd.concat([trips_per_route_weekday, trips_per_route_saturday, trips_per_route_sunday], ignore_index=True)
)


all_day_without_routes = merge_and_aggregate_without_routes(
    stops_without_route_all_day, 
    pd.concat([trips_per_route_weekday, trips_per_route_saturday, trips_per_route_sunday], ignore_index=True)
)

stops_aggregated_all_day = pd.concat(
    [all_day_with_routes, all_day_without_routes],
    ignore_index=True
)

In [40]:
stops_aggregated_weekday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_weekday.csv", index=False)
stops_aggregated_saturday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_saturday.csv", index=False)
stops_aggregated_sunday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_sunday.csv", index=False)
stops_aggregated_all_day.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_allday.csv", index=False)